In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from dx7pytorch.filters import filter_get_all_op_ratio
from IPython.display import Audio, display
from dx7pytorch import DXDataset
import torch.utils.data as data

def midi_to_note(midi_number):
    """convert midi note value to note name (60=C4)"""
    notes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    octave = (midi_number // 12) - 1
    note = notes[midi_number % 12]
    return f"{note}{octave}"

# --- Dataset setup ---
sr = 48000
collection_path = '../dataset/collection.bin'
dataset = DXDataset(
    sr,
    collection_path,
    valid_notes=[i for i in range (36,60)], #2 octaves
    valid_velocities=(127,),
    note_on_len=sr,
    note_off_len=sr,
    subsample_ratio=0.1,
    random_seed=1234,
    filter_function=filter_get_all_op_ratio,
    chord_probability=0.5 #50% chords
)

# Split dataset
n_train_examples = int(len(dataset)*0.7)
n_valid_examples = int(len(dataset)*0.2)
n_test_examples = len(dataset) - n_train_examples - n_valid_examples

train_data, valid_data, test_data = torch.utils.data.random_split(
    dataset, [n_train_examples, n_valid_examples, n_test_examples]
)

train_loader = data.DataLoader(train_data, batch_size=4, shuffle=True)

print(f"Dataset length: {len(dataset)}. Playing 5 synthesized batches...")

# --- Play a few examples ---
i = 0
for instance in train_loader:
    if i == 5:
        break
    i += 1
    
    audios = instance['audio']  # shape: [batch, channels, samples]
    names = instance['name']
    chords = instance['chord'] #get chord types
    notes = instance['note'] #get root notes
    
    for j in range(audios.shape[0]):
        audio = audios[j].numpy()
        name = names[j]
        chord_type = chords[j]
        root = notes[j].item()  # Convert tensor to int
        note_name = midi_to_note(root)
        
        # Convert from multi-channel (if stereo) to mono
        if audio.ndim > 1:
            audio = np.mean(audio, axis=0)
        
        #print what's being played
        if chord_type != '':  # it's a chord
            print(f"Patch: {name} | CHORD: {chord_type} rooted at {note_name}")
        else:  # Single note
            print(f"Patch: {name} | Single note: {note_name}")
        
        # plot waveform with better title
        plt.figure(figsize=(10, 3))
        if chord_type != '':
            plt.title(f"Waveform: {name} - {chord_type} (root: {note_name})")
        else:
            plt.title(f"Waveform: {name} - Note {note_name}")
        plt.plot(audio)
        plt.show()
        
        # Play audio in notebook
        display(Audio(audio, rate=sr))